# Subagents: Delegating to a Child Agent with Isolated Context

The SDK supports subagents — delegated child agents with their own context, isolated from the parent's conversation history. This is the same subagent machinery that powers Claude Code's own `Task` tool.


In [ ]:
from claude_agent_sdk import (
    ClaudeSDKClient,  # a client you can keep open and send several messages through
    ClaudeAgentOptions,  # settings object: model, system prompt, tools, etc.
    AgentDefinition,  # describes one subagent: what it's for and how it behaves
    AssistantMessage,  # message type that holds Claude's actual reply
    ToolUseBlock,  # message piece that shows "Claude is calling a tool now"
    ResultMessage,  # the last message in the stream — carries the final answer plus stats (cost, duration, etc.)
)

# AgentDefinition describes a "child" agent the main (parent) agent can hand
# work off to. Think of it as defining a specialist teammate:
summarizer = AgentDefinition(
    # description: tells the PARENT agent what this subagent is good for,
    # so it knows WHEN to delegate to it.
    description="Summarizes a block of text in exactly one sentence. Use this whenever the user needs a quick one-sentence summary.",
    # prompt: the subagent's OWN system prompt — its personality/instructions,
    # completely separate from the parent's.
    prompt="You summarize whatever text you're given in exactly one sentence. Nothing else.",
    # tools: which tools this subagent is allowed to use. Empty list = none —
    # it can only read what it's told and reply in text.
    tools=[],
    # background: False means the parent WAITS for this subagent to finish
    # and get its result back before continuing (a blocking call).
    background=False,
)

# agents={"summarizer": summarizer} registers our subagent under the name
# "summarizer" — the parent will refer to it by this name when delegating.
options = ClaudeAgentOptions(model="haiku", agents={"summarizer": summarizer})

## Delegate from a parent session — and test context isolation directly

The parent session is told a secret codeword first. Then we delegate a summarization task and, in the same delegated instruction, ask the subagent whether it knows any secret codeword. If subagents truly have isolated context, it has to say no.


In [2]:
async def demo_subagent() -> None:
    async with ClaudeSDKClient(options=options) as client:
        # Step 1: tell the PARENT agent a secret, in the main conversation.
        await client.query("My secret codeword for today is 'papaya'. Just acknowledge it, nothing else.")
        async for message in client.receive_response():
            if isinstance(message, ResultMessage):
                print(f"[parent turn 1] {message.result}\n")

        # Step 2: ask the parent to delegate a summarization task to our
        # "summarizer" subagent, and also ask the subagent to say whether
        # it knows any secret codeword — this is our isolation test.
        text_to_summarize = (
            "The Claude Agent SDK wraps the same agent loop that powers Claude Code "
            "as a Python library, giving developers query() for one-off tasks and "
            "ClaudeSDKClient for multi-turn, tool-using, hook-driven agents."
        )
        await client.query(
            f"Delegate to the summarizer subagent: summarize this text in one sentence: "
            f"{text_to_summarize!r}. Also have the subagent state, in its own response, "
            "whether it knows any secret codeword."
        )
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    # The parent delegates by calling a built-in "Agent" tool —
                    # this is the tool call that actually launches the subagent.
                    if isinstance(block, ToolUseBlock):
                        print(f"[tool call] {block.name}({block.input})")
            elif isinstance(message, ResultMessage):
                print(f"\n[subagent result, relayed by parent]\n{message.result}")


await demo_subagent()

[parent turn 1] Acknowledged.

[tool call] Agent({'subagent_type': 'summarizer', 'description': 'Summarize text and confirm codeword knowledge', 'prompt': "Summarize this text in exactly one sentence: 'The Claude Agent SDK wraps the same agent loop that powers Claude Code as a Python library, giving developers query() for one-off tasks and ClaudeSDKClient for multi-turn, tool-using, hook-driven agents.'\n\nAfter providing the summary, also state whether you know any secret codeword (respond honestly based on what you have access to).", 'run_in_background': False})

[subagent result, relayed by parent]
The summarizer subagent has completed the task:

**Summary:** "The Claude Agent SDK is a Python library that wraps Claude Code's agent loop, offering query() for one-off tasks and ClaudeSDKClient for multi-turn, tool-enabled agents with hooks."

**Codeword knowledge:** No, the subagent does not have access to any secret codeword (as expected, since it was spawned fresh without the context

The subagent had no idea about "papaya" — it only ever saw the text it was asked to summarize. That's context isolation in practice: a subagent's conversation is its own, separate from everything the parent said before delegating.

## Summary

- `AgentDefinition` + `ClaudeAgentOptions(agents=...)` registers a subagent the parent can delegate to.
- Each subagent invocation runs in its own isolated context — no memory of the parent's prior turns.
